In [2]:
import numpy as np
import pandas as pd
import joblib
import pickle

print("LOADING MODEL...")

# Try multiple methods to load model
model = None

# Method 1: Try joblib
try:
    model = joblib.load('./models_saved/best_model.pkl')
    print("Model loaded with joblib")
except Exception as e:
    print(f"Joblib failed: {e}")

# Method 2: Try pickle
if model is None or isinstance(model, str):
    try:
        with open('./models_saved/best_model.pkl', 'rb') as f:
            model = pickle.load(f)
        print("Model loaded with pickle")
    except Exception as e:
        print(f"Pickle failed: {e}")

# Method 3: Try alternative models
if isinstance(model, str):
    print(f"Model is string: {model}")
    alt_models = [
        './models_saved/random_forest.pkl',
        './models_saved/xgboost.pkl',
        './models_saved/lightgbm.pkl',
        './models_saved/voting_ensemble.pkl'
    ]
    
    for alt in alt_models:
        try:
            model = joblib.load(alt)
            print(f"Loaded alternative: {alt}")
            break
        except:
            continue

# Method 4: Use in-memory model if available
if model is None or isinstance(model, str):
    if 'best_model' in globals() and not isinstance(globals()['best_model'], str):
        model = globals()['best_model']
        print("Using global best_model")
    elif 'rf_model' in globals() and globals()['rf_model'] is not None:
        model = globals()['rf_model']
        print("Using rf_model from memory")
    elif 'xgb_model' in globals() and globals()['xgb_model'] is not None:
        model = globals()['xgb_model']
        print("Using xgb_model from memory")

# Final validation
if model is None or isinstance(model, str):
    print("ERROR: Could not load any valid model!")
    exit()

if not hasattr(model, 'predict'):
    print(f"ERROR: Model has no predict method! Type: {type(model)}")
    exit()

print(f"Model loaded successfully! Type: {type(model).__name__}")
print("")

# ============================================================================
# FRAUD SAMPLES
# ============================================================================

# Sample 1: Transfer Fraud
fraud_sample_1 = {
    'step': 1,
    'amount': 181.00,
    'type_encoded': 4,
    'hour_of_day': 1,
    'day_of_week': 0,
    'week_of_month': 1,
    'log_amount': np.log1p(181.00),
    'sqrt_amount': np.sqrt(181.00),
    'zscore_amount': (181.00 - 179861.90) / 603858.23,
    'amount_vs_type_avg': 181.00 / 50000,
    'amount_vs_step_avg': 181.00 / 40000,
    'is_high_value': 0,
    'is_very_high_value': 0,
    'amount_count': 1,
    'is_duplicate_amount': 0,
    'is_cash_out': 0,
    'is_transfer': 1,
    'is_payment': 0,
    'is_cash_in': 0,
    'type_frequency': 0.08,
    'amount_flagged_interaction': 0,
    'transfer_flagged': 0,
    'cashout_flagged': 0,
    'isFlaggedFraud': 0
}

# Sample 2: Cash Out Fraud
fraud_sample_2 = {
    'step': 2,
    'amount': 500000.00,
    'type_encoded': 1,
    'hour_of_day': 3,
    'day_of_week': 6,
    'week_of_month': 2,
    'log_amount': np.log1p(500000),
    'sqrt_amount': np.sqrt(500000),
    'zscore_amount': (500000 - 179861.90) / 603858.23,
    'amount_vs_type_avg': 500000 / 40000,
    'amount_vs_step_avg': 500000 / 45000,
    'is_high_value': 1,
    'is_very_high_value': 1,
    'amount_count': 1,
    'is_duplicate_amount': 0,
    'is_cash_out': 1,
    'is_transfer': 0,
    'is_payment': 0,
    'is_cash_in': 0,
    'type_frequency': 0.35,
    'amount_flagged_interaction': 0,
    'transfer_flagged': 0,
    'cashout_flagged': 0,
    'isFlaggedFraud': 0
}

# Sample 3: Large Transfer at Night
fraud_sample_3 = {
    'step': 4,
    'amount': 750000.00,
    'type_encoded': 4,
    'hour_of_day': 4,
    'day_of_week': 0,
    'week_of_month': 1,
    'log_amount': np.log1p(750000),
    'sqrt_amount': np.sqrt(750000),
    'zscore_amount': (750000 - 179861.90) / 603858.23,
    'amount_vs_type_avg': 750000 / 50000,
    'amount_vs_step_avg': 750000 / 35000,
    'is_high_value': 1,
    'is_very_high_value': 1,
    'amount_count': 1,
    'is_duplicate_amount': 0,
    'is_cash_out': 0,
    'is_transfer': 1,
    'is_payment': 0,
    'is_cash_in': 0,
    'type_frequency': 0.08,
    'amount_flagged_interaction': 0,
    'transfer_flagged': 0,
    'cashout_flagged': 0,
    'isFlaggedFraud': 0
}

# Sample 4: Duplicate Amount Pattern
fraud_sample_4 = {
    'step': 10,
    'amount': 25000.00,
    'type_encoded': 4,
    'hour_of_day': 23,
    'day_of_week': 5,
    'week_of_month': 4,
    'log_amount': np.log1p(25000),
    'sqrt_amount': np.sqrt(25000),
    'zscore_amount': (25000 - 179861.90) / 603858.23,
    'amount_vs_type_avg': 25000 / 50000,
    'amount_vs_step_avg': 25000 / 30000,
    'is_high_value': 0,
    'is_very_high_value': 0,
    'amount_count': 5,
    'is_duplicate_amount': 1,
    'is_cash_out': 0,
    'is_transfer': 1,
    'is_payment': 0,
    'is_cash_in': 0,
    'type_frequency': 0.08,
    'amount_flagged_interaction': 0,
    'transfer_flagged': 0,
    'cashout_flagged': 0,
    'isFlaggedFraud': 0
}


print("FRAUD DETECTION TEST - SAMPLES")


fraud_samples = [
    ("Transfer Fraud (Small Amount)", fraud_sample_1),
    ("Cash Out Fraud (Large Amount)", fraud_sample_2),
    ("Large Transfer at Night", fraud_sample_3),
    ("Duplicate Amount Pattern", fraud_sample_4)
]

for name, sample in fraud_samples:
    sample_df = pd.DataFrame([sample])
    
    try:
        pred = model.predict(sample_df)[0]
        proba = model.predict_proba(sample_df)[0][1] if hasattr(model, 'predict_proba') else None
        
        print(f"\n{name}")
        print(f"  Amount: ${sample['amount']:,.2f}")
        type_names = ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
        print(f"  Type: {type_names[sample['type_encoded']]}")
        print(f"  Hour: {sample['hour_of_day']}:00")
        print(f"  Prediction: {'FRAUD' if pred == 1 else 'NON-FRAUD'}")
        if proba is not None:
            print(f"  Probability: {proba:.4f}")
            risk = 'HIGH' if proba > 0.8 else 'MEDIUM' if proba > 0.5 else 'LOW'
            print(f"  Risk Level: {risk}")
    except Exception as e:
        print(f"Error predicting {name}: {e}")


print("TEST COMPLETED")

LOADING MODEL...
Model loaded with joblib
Model loaded with pickle
Model is string: XGBoost
Loaded alternative: ./models_saved/random_forest.pkl
Model loaded successfully! Type: RandomForestClassifier

FRAUD DETECTION TEST - SAMPLES

Transfer Fraud (Small Amount)
  Amount: $181.00
  Type: TRANSFER
  Hour: 1:00
  Prediction: NON-FRAUD
  Probability: 0.0221
  Risk Level: LOW

Cash Out Fraud (Large Amount)
  Amount: $500,000.00
  Type: CASH_OUT
  Hour: 3:00
  Prediction: NON-FRAUD
  Probability: 0.0400
  Risk Level: LOW

Large Transfer at Night
  Amount: $750,000.00
  Type: TRANSFER
  Hour: 4:00
  Prediction: NON-FRAUD
  Probability: 0.0403
  Risk Level: LOW

Duplicate Amount Pattern
  Amount: $25,000.00
  Type: TRANSFER
  Hour: 23:00
  Prediction: NON-FRAUD
  Probability: 0.3667
  Risk Level: LOW
TEST COMPLETED
